<!-- nav -->
[← A module and its interfaces](01_A_Module_And_Its_Interfaces.ipynb) · [Home](../../README.md) · [Organizing a design →](03_Organizing_A_Design.ipynb)

# Implementing a module

A module says where it meets the world; an *implementation* says how it works inside. An implementation realizes one
module with *instances* of other modules, which run asynchronously and cooperate only by exchanging messages over
*channels*. A channel is directional: it carries messages of one schema from its sources to its destinations.

This tutorial builds a small signal pipeline: a sensor takes samples, a filter smooths them, and a logger records
them, while the pipeline also hands its filtered samples to whatever uses it.

## The modules, and what their messages are

Messages are data, so their schema is an mbse-schemas schema like any other: here a value object, `dsp.Sample`. The
modules are built as in the [first tutorial](01_A_Module_And_Its_Interfaces.ipynb), through a helper that takes
interfaces and ports as dictionaries.

In [1]:
from mbse.Schemas.Framework import Builders
from mbse.Architecture.Systems.Framework import Interactions

store = Builders.Store()
I = Interactions.define(store)
library = store.singleton("Interactions.Library")

with store.transaction():
    Sample = store.Schemas.OfObject(lambda o: o.name("dsp.Sample").properties([
        lambda t: t.name("time").schema([store.Schemas.OfFloat]),
        lambda t: t.name("value").schema([store.Schemas.OfFloat]),
    ]))


def module(name, interfaces):
    """A module in the library, with interfaces by name, each with ports by name and direction."""
    def interface(ports):
        return lambda i: i.ports(lambda a: [
            a.insert(lambda e, n=n, d=d: e.name(n).port(lambda p: p.direction(d))) for n, d in ports.items()])

    store.Interactions.Library(library, lambda lib: lib.modules(lambda a: a.insert(lambda e: e.name(name).module(
        lambda m: m.interfaces(lambda a: [
            a.insert(lambda e, n=n, p=p: e.name(n).intf(interface(p))) for n, p in interfaces.items()])))))
    return library.modules.at(name).module


with store.transaction():
    sensor = module("Sensor", {"samples": {"tx": "out"}})
    filter_ = module("Filter", {"samples": {"rx": "in"}, "filtered": {"tx": "out"}})
    logger = module("Logger", {"records": {"rx": "in"}})
    pipeline = module("Pipeline", {"readings": {"out": "out"}})
print([str(e.name) for e in library.modules])

['Sensor', 'Filter', 'Logger', 'Pipeline']


## An implementation, its instances and channels

The implementation `reference` implements `Pipeline` (`Interactions.Implements`). It holds three instances by name
(`Interactions.HasInstance`), each of one module (`Interactions.Instantiates`), and two channels by name
(`Interactions.HasChannel`), each carrying `dsp.Sample`s. A channel links the schema of its messages, the schema
object itself, through `Interactions.HasSchema`.

In [2]:
def instance_of(m):
    return lambda i: i.module(lambda a: a.insert(lambda e: e.module(m)))


def carrying(schema):
    return lambda c: c.message(lambda a: a.insert(lambda e: e.schema(schema)))


with store.transaction():
    impl = store.Interactions.Implementation(lambda x: (
        x
        .name("reference")
        .module(lambda a: a.insert(lambda e: e.module(pipeline)))
        .instances(lambda a: (
            a
            .insert(lambda e: e.name("sensor").instance(instance_of(sensor)))
            .insert(lambda e: e.name("filter").instance(instance_of(filter_)))
            .insert(lambda e: e.name("logger").instance(instance_of(logger)))
        ))
        .channels(lambda a: (
            a
            .insert(lambda e: e.name("raw").channel(carrying(Sample)))
            .insert(lambda e: e.name("filtered").channel(carrying(Sample)))
        ))
    ))

raw, filtered = impl.channels.at("raw").channel, impl.channels.at("filtered").channel
print(Interactions.module_of(impl.instances.at("filter").instance) is filter_)
print(Interactions.message_schema(raw).name, Interactions.distribution(raw))

True
dsp.Sample BROADCAST


## Attaching channels

`Interactions.Connects(channel, instance, intf, port)` attaches a channel to one port of one interface of an instance.
The port's direction gives the end it makes: an `"out"` port sends, so it is a *source*; an `"in"` port receives, so
it is a *destination*.

At the edge of the implementation, a channel also reaches the module being implemented. There the implementation
itself fills `instance`, standing for its module's boundary, and directions turn around: the pipeline's `readings.out`
port sends to the outside world, so inside the implementation it *receives* from the channel, a destination.

In [3]:
def connect(channel, at, intf, port):
    interface = Interactions.module_of(at).interfaces.at(intf).intf
    store.Interactions.Channel(channel, lambda c: c.connections(lambda a: a.insert(
        lambda e: e.instance(at).intf(interface).port(interface.ports.at(port).port))))


with store.transaction():
    connect(raw, impl.instances.at("sensor").instance, "samples", "tx")
    connect(raw, impl.instances.at("filter").instance, "samples", "rx")
    connect(filtered, impl.instances.at("filter").instance, "filtered", "tx")
    connect(filtered, impl.instances.at("logger").instance, "records", "rx")
    connect(filtered, impl, "readings", "out")

for end in Interactions.ends(filtered):
    print(f"{end.role:12} {Interactions.path(end.instance):18} {Interactions.path(end.port)}")

source       reference.filter   Filter.filtered.tx
destination  reference.logger   Logger.records.rx
destination  reference          Pipeline.readings.out


## How a message reaches several destinations

`filtered` has two destinations. Its `distribution` says how each message reaches them: `"BROADCAST"`, to every one,
which is what a channel does when it says nothing; `"ANYCAST"`, to any one; or `"ROUND_ROBIN"`, to each in turn.

In [4]:
with store.transaction():
    store.Interactions.Channel(filtered, lambda c: c.distribution("BROADCAST"))
print(Interactions.distribution(filtered), Interactions.DISTRIBUTIONS)

BROADCAST ('BROADCAST', 'ANYCAST', 'ROUND_ROBIN')


## Is it right?

`Interactions.Check()` lists what the schemas' own constraints cannot say: a channel with no source, an attachment to
an interface the instance's module lacks, a port without a direction, and so on. Here there is nothing to report.
[Checking and saving](04_Checking_And_Saving.ipynb) shows what it finds when there is.

In [5]:
print(Interactions.Check()(pipeline))

[]


The whole pipeline, built this way and a little more, is `mbse.Architecture.Systems.Examples.Pipeline`, whose
`build(store)` the next tutorials start from. Next, [Organizing a design](03_Organizing_A_Design.ipynb) groups its
parts without changing its structure.

<!-- nav -->
[← A module and its interfaces](01_A_Module_And_Its_Interfaces.ipynb) · [Home](../../README.md) · [Organizing a design →](03_Organizing_A_Design.ipynb)